# 🎮 PixelZone Gaming Café — Big Data Analytics Project

**Module:** IBM4208 – Big Data Analytics  
**Institution:** INTI International University  
**Programme:** BCSI  

---

## Project Overview

This notebook covers the full Big Data Analytics pipeline for **PixelZone Gaming Café**, a fictional high-end gaming café concept based in Kuala Lumpur, Malaysia.

### What this notebook does:
1. **Dataset Generation** — Synthetically generates 50,000 realistic customer session records (2021–2025) using business-realistic patterns (peak hours, membership tiers, seasonal effects)
2. **Data Validation & Cleaning** — Verifies data quality and prepares it for analysis
3. **Python Visualizations** — 3 business-insight charts (revenue trend, genre popularity, ratings distribution)
4. **Machine Learning** — Linear Regression vs Random Forest Regressor to predict monthly revenue
5. **Forecast** — 2026 revenue projection using the best-performing model

### Key Result
> Linear Regression achieved **R² = 0.9838** (98.4% accuracy) on the monthly revenue prediction task, outperforming Random Forest (R² = 0.7880).

---

## Part 1 — Dataset Generation

Generates a synthetic 50,000-row dataset simulating 5 years of PixelZone customer sessions (Jan 2021 – Dec 2025). Business-realistic patterns are embedded:
- Peak demand: 6PM–2AM, weighted toward weekends
- ~5% year-on-year revenue growth
- Membership-based pricing (Basic / Silver / Gold) and zone (Casual / VIP)
- Seasonal food combo patterns (holiday months: June, July, December)
- Age distribution centred on 21 (core student demographic)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

print("Libraries imported successfully.")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RANDOM_SEED  = 42
START_DATE   = datetime(2021, 1, 1)
END_DATE     = datetime(2025, 12, 31)
TARGET_ROWS  = 50_000

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ── Categorical value pools ───────────────────────────────────────────────────
GAME_GENRES      = ['FPS', 'RPG', 'Sports', 'Strategy', 'Racing', 'VR']
PAYMENT_METHODS  = ['Cash', 'Card', 'E-Wallet']
FOOD_COMBOS      = ['None', 'Solo Snack', 'Solo Meal',
                    'Duo Deal', 'Duo Meal', 'Trio Deal', 'Party Bundle']
MEMBERSHIP_TYPES = ['Basic', 'Silver', 'Gold']
ZONES            = ['Casual', 'VIP']
TIME_PERIODS     = ['6AM-12PM', '12PM-2PM', '2PM-6PM',
                    '6PM-10PM', '10PM-2AM', '2AM-6AM']

print(f"Date range : {START_DATE.date()} → {END_DATE.date()}")
print(f"Target rows: {TARGET_ROWS:,}")

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def get_hourly_rate(membership: str, zone: str) -> float:
    """Return an hourly rate (MYR) based on membership tier and zone."""
    rate_map = {
        ('VIP',    'Gold'):   (10, 13),
        ('VIP',    'Silver'): (8,  10),
        ('VIP',    'Basic'):  (6,   8),
        ('Casual', 'Gold'):   (5,   7),
        ('Casual', 'Silver'): (4,   6),
        ('Casual', 'Basic'):  (3,   5),
    }
    lo, hi = rate_map[(zone, membership)]
    return round(random.uniform(lo, hi), 2)


def get_time_period(day: str) -> str:
    """Return a time slot; weekends skew later into the night."""
    weekend = day in ('Friday', 'Saturday', 'Sunday')
    weights = [5, 10, 20, 25, 30, 10] if weekend else [5, 15, 25, 30, 20, 5]
    return random.choices(TIME_PERIODS, weights=weights, k=1)[0]


def get_rating(membership: str) -> int:
    """Higher-tier members tend to rate more generously."""
    rating_map = {
        'Gold':   ([3, 4, 5],    [10, 35, 55]),
        'Silver': ([2, 3, 4, 5], [5,  20, 45, 30]),
        'Basic':  ([1, 2, 3, 4, 5], [5, 15, 35, 30, 15]),
    }
    values, weights = rating_map[membership]
    return random.choices(values, weights=weights, k=1)[0]


YEARLY_GROWTH = {2021: 1.00, 2022: 1.05, 2023: 1.10, 2024: 1.15, 2025: 1.20}


def is_repeat_customer(membership: str) -> str:
    """Gold members are far more likely to be repeat customers."""
    repeat_weights = {'Gold': [80, 20], 'Silver': [60, 40], 'Basic': [35, 65]}
    return random.choices(['Yes', 'No'], weights=repeat_weights[membership], k=1)[0]


print("Helper functions defined.")

In [ ]:
# ── Generate dataset ──────────────────────────────────────────────────────────
total_days = (END_DATE - START_DATE).days
rows = []

for i in range(TARGET_ROWS):
    current_date = START_DATE + timedelta(days=random.randint(0, total_days))
    day_name     = current_date.strftime('%A')
    year, month  = current_date.year, current_date.month

    membership = random.choices(MEMBERSHIP_TYPES, weights=[50, 35, 15], k=1)[0]
    zone       = random.choices(ZONES, weights=[30, 70] if membership == 'Gold' else [70, 30], k=1)[0]

    hourly_rate  = get_hourly_rate(membership, zone) * YEARLY_GROWTH.get(year, 1.0)
    hourly_rate  = round(hourly_rate, 2)

    hours_played = round(
        random.choices([1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5],
                       weights=[10, 15, 25, 20, 15, 8, 4, 2, 1], k=1)[0], 1
    )

    # Seasonal food combo bias (holidays: June, July, December)
    if month in (6, 7, 12):
        food_weights = [15, 15, 15, 20, 15, 15, 5]
    else:
        food_weights = [35, 15, 15, 15, 10, 8, 2]

    rows.append({
        'Session_ID':       f'PZ-{i + 1:05d}',
        'Date':             current_date.strftime('%Y-%m-%d'),
        'Day_of_Week':      day_name,
        'Time_Period':      get_time_period(day_name),
        'Customer_ID':      f'CUS-{random.randint(1000, 9999)}',
        'Age':              int(np.clip(np.random.normal(loc=21, scale=5), 13, 50)),
        'Membership_Type':  membership,
        'Zone':             zone,
        'Game_Genre':       random.choice(GAME_GENRES),
        'Hours_Played':     hours_played,
        'Hourly_Rate_MYR':  hourly_rate,
        'Total_Amount_MYR': round(hours_played * hourly_rate, 2),
        'Payment_Method':   random.choice(PAYMENT_METHODS),
        'Food_Combo':       random.choices(FOOD_COMBOS, weights=food_weights, k=1)[0],
        'Rating':           get_rating(membership),
        'Repeat_Customer':  is_repeat_customer(membership),
    })

df = (pd.DataFrame(rows)
        .sort_values('Date')
        .reset_index(drop=True))

# Re-assign sequential Session IDs after sort
df['Session_ID'] = [f'PZ-{i + 1:05d}' for i in range(len(df))]

print(f"Dataset generated: {len(df):,} rows × {len(df.columns)} columns")
df.head(5)

## Part 2 — Data Validation & Export

In [ ]:
# ── Validation summary ────────────────────────────────────────────────────────
print("=" * 50)
print(" DATASET VALIDATION")
print("=" * 50)
print(f"Shape            : {df.shape}")
print(f"Date range       : {df['Date'].min()} → {df['Date'].max()}")
print(f"Missing values   : {df.isnull().sum().sum()}")
print(f"Avg daily sessions: {len(df) / 1826:.0f}")
print()
print("Membership distribution (%)")
print((df['Membership_Type'].value_counts(normalize=True) * 100).round(1))
print()
print("Yearly Revenue (MYR)")
df['Year'] = pd.to_datetime(df['Date']).dt.year
print(df.groupby('Year')['Total_Amount_MYR'].sum().round(2))
df.drop(columns='Year', inplace=True)

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────────────
OUTPUT_CSV = '../data/pixelzone_dataset.csv'
df.to_csv(OUTPUT_CSV, index=False)
print(f"Dataset saved → {OUTPUT_CSV}")

## Part 3 — Python-Based Data Visualization

Three charts are produced to derive business insights from the generated dataset:

| Chart | Type | Business Question |
|-------|------|------------------|
| 1 | Line | How has monthly revenue trended over 5 years? |
| 2 | Bar  | Which game genres are most popular? |
| 3 | Pie  | How are customer satisfaction ratings distributed? |

In [ ]:
# ── Visualization imports & setup ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

# Prepare time columns
df['Date']      = pd.to_datetime(df['Date'])
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M')

print("Visualization setup complete.")

In [ ]:
# ── Chart 1: Monthly Revenue Trend (2021–2025) ────────────────────────────────
monthly_rev = df.groupby('YearMonth')['Total_Amount_MYR'].sum().reset_index()
monthly_rev['YearMonth_str'] = monthly_rev['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(monthly_rev['YearMonth_str'], monthly_rev['Total_Amount_MYR'],
        color='#2196F3', linewidth=2, marker='o', markersize=3)

for year in range(2021, 2026):
    ax.axvline(x=f'{year}-01', color='gray', linestyle='--', alpha=0.35, linewidth=1)
    ax.text(f'{year}-01', monthly_rev['Total_Amount_MYR'].max() * 0.96,
            str(year), fontsize=9, color='gray')

ax.set_title('PixelZone Gaming Café — Monthly Revenue Trend (2021–2025)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Total Revenue (MYR)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xticks(monthly_rev['YearMonth_str'][::6])
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/charts/chart1_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 1 saved.")

In [ ]:
# ── Chart 2: Game Genre Popularity ───────────────────────────────────────────
genre_counts = df['Game_Genre'].value_counts().reset_index()
genre_counts.columns = ['Game_Genre', 'Session_Count']

colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(genre_counts))]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(genre_counts['Game_Genre'], genre_counts['Session_Count'],
              color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, genre_counts['Session_Count']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('PixelZone Gaming Café — Game Genre Popularity',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Game Genre', fontsize=11)
ax.set_ylabel('Number of Sessions', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/charts/chart2_genre_popularity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 2 saved.")

In [ ]:
# ── Chart 3: Customer Ratings Distribution ────────────────────────────────────
rating_counts = df['Rating'].value_counts().sort_index()
labels  = ['1 Star', '2 Stars', '3 Stars', '4 Stars', '5 Stars']
colors  = ['#EF5350', '#FF8A65', '#FFF176', '#81C784', '#2196F3']
explode = (0, 0, 0, 0.05, 0.05)

fig, ax = plt.subplots(figsize=(8, 7))
wedges, texts, autotexts = ax.pie(
    rating_counts, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=140, explode=explode,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')

ax.set_title('PixelZone Gaming Café — Customer Ratings Distribution',
             fontsize=14, fontweight='bold', pad=20)
ax.annotate(f'Total Sessions: {len(df):,}', xy=(0, -1.3),
            ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('../outputs/charts/chart3_ratings.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 3 saved.")

## Part 4 — Machine Learning: Revenue Prediction

The 50,000 session records are aggregated into **60 monthly summaries** (Jan 2021 – Dec 2025). Two regression models are trained and compared:

- **Linear Regression** — suited to the consistently upward revenue trend
- **Random Forest Regressor** — ensemble of 100 decision trees

**Target variable:** `Total_Monthly_Revenue (MYR)`  
**Train/test split:** 80% / 20% (48 months training, 12 months testing)

In [ ]:
# ── ML imports ────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("ML libraries ready.")

In [ ]:
# ── Feature engineering: aggregate to monthly level ───────────────────────────
monthly_df = (df.groupby(['Year', 'Month'])
                .agg(
                    Total_Revenue   = ('Total_Amount_MYR', 'sum'),
                    Avg_Hours       = ('Hours_Played',     'mean'),
                    Avg_Rating      = ('Rating',           'mean'),
                    Session_Count   = ('Session_ID',       'count'),
                    Avg_Hourly_Rate = ('Hourly_Rate_MYR',  'mean'),
                )
                .reset_index())

monthly_df['Month_Number'] = range(1, len(monthly_df) + 1)

FEATURES = ['Year', 'Month', 'Month_Number',
            'Avg_Hours', 'Avg_Rating', 'Session_Count', 'Avg_Hourly_Rate']

X = monthly_df[FEATURES]
y = monthly_df['Total_Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Monthly records : {len(monthly_df)}")
print(f"Training months : {len(X_train)}")
print(f"Testing  months : {len(X_test)}")

In [ ]:
# ── Model 1: Linear Regression ────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_metrics = {
    'MAE':  mean_absolute_error(y_test, lr_pred),
    'MSE':  mean_squared_error(y_test, lr_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, lr_pred)),
    'R²':   r2_score(y_test, lr_pred),
}

# ── Model 2: Random Forest ────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_metrics = {
    'MAE':  mean_absolute_error(y_test, rf_pred),
    'MSE':  mean_squared_error(y_test, rf_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, rf_pred)),
    'R²':   r2_score(y_test, rf_pred),
}

# ── Comparison table ──────────────────────────────────────────────────────────
comparison = pd.DataFrame({'Linear Regression': lr_metrics,
                           'Random Forest':     rf_metrics})
print(comparison.to_string(float_format=lambda x: f'{x:,.4f}'))
print(f"\nWinner: {'Linear Regression' if lr_metrics['R²'] > rf_metrics['R²'] else 'Random Forest'}")

In [ ]:
# ── Residual analysis (best model: Linear Regression) ─────────────────────────
residuals = y_test.values - lr_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PixelZone — Residual Analysis (Linear Regression)',
             fontsize=13, fontweight='bold')

ax1.scatter(lr_pred, residuals, alpha=0.7, color='#2196F3', edgecolors='white')
ax1.axhline(0, color='red', linestyle='--', linewidth=1.2)
ax1.set_xlabel('Predicted Revenue (MYR)'); ax1.set_ylabel('Residuals (MYR)')
ax1.set_title('Residuals vs Predicted Values')

ax2.hist(residuals, bins=10, color='#2196F3', edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', linewidth=1.2)
ax2.set_xlabel('Residual Value (MYR)'); ax2.set_ylabel('Frequency')
ax2.set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('../outputs/charts/chart4_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean residual : MYR {residuals.mean():.2f}")
print(f"Std  residual : MYR {residuals.std():.2f}")

In [ ]:
# ── Actual vs Predicted comparison plot ───────────────────────────────────────
sort_idx        = y_test.argsort()
y_sorted        = y_test.iloc[sort_idx].values
lr_pred_sorted  = lr_pred[sort_idx]
rf_pred_sorted  = rf_pred[sort_idx]
x_axis          = range(len(y_sorted))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PixelZone — Model Comparison: Actual vs Predicted Revenue',
             fontsize=13, fontweight='bold')

for ax, preds, label, color in [
    (ax1, lr_pred_sorted, 'Linear Regression', '#E53935'),
    (ax2, rf_pred_sorted, 'Random Forest',      '#43A047'),
]:
    ax.plot(x_axis, y_sorted, 'b-o', label='Actual', markersize=5)
    ax.plot(x_axis, preds, linestyle='--', marker='s', color=color,
            label='Predicted', markersize=5)
    ax.set_title(label); ax.set_xlabel('Test Samples')
    ax.set_ylabel('Monthly Revenue (MYR)')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/charts/chart5_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5 — 2026 Revenue Forecast

The selected Linear Regression model projects monthly revenue for all 12 months of 2026, assuming a **5% growth** in session count and average hourly rate.

In [ ]:
# ── Build future feature rows ─────────────────────────────────────────────────
last = monthly_df.iloc[-1]

future_rows = pd.DataFrame([{
    'Year':           2026,
    'Month':          m,
    'Month_Number':   60 + m,
    'Avg_Hours':      last['Avg_Hours'],
    'Avg_Rating':     last['Avg_Rating'],
    'Session_Count':  int(last['Session_Count'] * 1.05),
    'Avg_Hourly_Rate': last['Avg_Hourly_Rate'] * 1.05,
} for m in range(1, 13)])

forecast_2026 = lr.predict(future_rows[FEATURES])

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

hist_2025 = monthly_df[monthly_df['Year'] == 2025]

# ── Forecast chart ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(range(1, 13), hist_2025['Total_Revenue'].values,
        'b-o', label='Actual 2025', linewidth=2, markersize=6)
ax.plot(range(1, 13), forecast_2026,
        color='#FF5722', linestyle='--', marker='s',
        label='Forecast 2026', linewidth=2, markersize=6)

for i, val in enumerate(forecast_2026):
    ax.annotate(f'MYR\n{val:,.0f}', xy=(i + 1, val),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=7, color='#FF5722')

ax.set_title('PixelZone Gaming Café — Revenue Forecast 2026\n(Linear Regression Model)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Monthly Revenue (MYR)', fontsize=11)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(month_names)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=11); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/charts/chart6_forecast_2026.png', dpi=150, bbox_inches='tight')
plt.show()

total_forecast = forecast_2026.sum()
print(f"\nProjected 2026 Annual Revenue: MYR {total_forecast:,.2f}")
print(f"vs Actual 2025 Annual Revenue: MYR {hist_2025['Total_Revenue'].sum():,.2f}")
print(f"Year-over-year growth : {((total_forecast / hist_2025['Total_Revenue'].sum()) - 1) * 100:.1f}%")

---

## Summary

| Stage | Output | Key Finding |
|-------|--------|-------------|
| Dataset | 50,000 rows × 16 columns | 5 years of realistic session data |
| Visualization | 3 business charts | Steady revenue growth; balanced genre demand |
| ML — Linear Regression | R² = **0.9838** | Best model; predicts within ~MYR 112/month |
| ML — Random Forest | R² = 0.7880 | Over-complex for this linear trend |
| Forecast | MYR ~174,300 (2026) | ~2.1% growth over 2025 |

**Decision:** The Cost-Benefit Analysis (in the full report) recommends a 5-setup expansion with a ~19-month payback period, generating MYR 2,025/month net benefit post-payback.